**Select chat model**

In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

# Load the .env file
load_dotenv()
# assign key from env to langchain/openai


True

**OpenAI llm**

In [9]:

from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage

print("API_KEY: ", os.getenv("OPENAI_API_KEY"))

# Make sure your API key is set in the environment
api_key = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = api_key

# Initialize model
llm = ChatOpenAI(
    model="gpt-4.1-nano",  # Make sure this model name is valid in your OpenAI account
    base_url="https://openrouter.ai/api/v1",
    temperature=0.6,
    api_key=api_key
)

# Create message list
messages = [
    SystemMessage(content="Say Hello in German"),
    HumanMessage(content="Can you speak German?")
]

# Call the model
response = llm.invoke(messages)

# Output result
print(response.content)

API_KEY:  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
Hallo! Ja, ich kann auf Deutsch sprechen.


**MistralAI llm**

In [14]:
os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")

model = init_chat_model("mistral-small", model_provider="mistralai", temperature=0.2)

from langchain_mistralai import ChatMistralAI
llm = ChatMistralAI(model="mistral-small")

response = model.invoke("Say hello in German")
response.usage_metadata
print(response)

content='The word "hello" can be translated to "hallo" in German. So, if you want to say hello in German, you can simply say "hallo"! This is a versatile and casual greeting that can be used in many different situations.\n\nHowever, it\'s worth noting that there are many other ways to say hello in German, depending on the time of day, the level of formality, and the relationship between the speakers. For example, "guten Tag" is a more formal way to say hello, and is often used in business settings or when meeting someone for the first time. "Guten Morgen" means "good morning," while "guten Abend" means "good evening."\n\nSo while "hallo" is a great way to say hello in many casual situations, it\'s always good to have a few other greetings up your sleeve as well!' additional_kwargs={} response_metadata={'token_usage': {'prompt_tokens': 13, 'total_tokens': 209, 'completion_tokens': 196}, 'model_name': 'mistral-small', 'model': 'mistral-small', 'finish_reason': 'stop'} id='run--4f965625-b

**invoke structured prompt**
- generate n idioms in different german level

In [ ]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative german Teacher. You generate german idioms accordings to given topic for given level"),
    ("human", "Provide {number_idioms} idioms with {topic} for level A2"),
])
messages = prompt.format_messages(number_idioms=5, topic="food")

response = model.invoke(messages)
print(response.content)

### Generate idioms with given user's input, and put up a question

In [ ]:
# Using PromptTemplate to set up prompt
from langchain.prompts import PromptTemplate
# call chain functions
from langchain.chains import LLMChain
# define model -> is done above

# define and chain idiom template
idiom_prompt = PromptTemplate(
    input_variables=["nbr_idioms", "topic", "level"],
    template =(
    "You are a creative German teacher. You generate German idioms according to a given topic for a given level.\n\n"
    "Provide {nbr_idioms} idioms about {topic} for level {level}."
    "After providing reponse ask user to make an example with one of given idiom"
    )
)
idiom_chain = LLMChain(llm=llm, prompt=idiom_prompt)

# fetch user's input
nbr_idioms = input("Enter number: ")
topic = input("Enter topic: ")
level = input("Enter level: ")

# collect user input
user_input = {
    "nbr_idioms": nbr_idioms,
    "topic": topic,
    "level": level
}

# run and print the chain
idiom_response = idiom_chain.run(user_input)
print("\nGenerated idioms: \n", idiom_response)



Generated idioms: 
 Sure, here are three German idioms about animals for level A1:

1. Das ist ein elefantengrünes Geschenkpapier - This is an elephant-green wrapping paper (meaning: It's a very obvious lie)
2. Wie die Lemminge ins Meer stürzen - To plunge into the sea like lemmings (meaning: To do something without thinking, often in large groups)
3. Ein schwarzes Schaf in der Familie sein - To be the black sheep in the family (meaning: To be the odd one out in a group or family)

Which idiom would you like to use to make an example sentence?


### generate evaluation based on the answer given by user

In [19]:
# from langchain.prompts import PromptTemplate
# define and chain evaluation_prompt
evaluation_prompt = PromptTemplate(
    input_variable = ["user_example", "idiom"],
    template=(
        "You are a German language teacher. Evaluate following sentece:\n"
        "\"{user_example}\"\n"
        "Did the user correctly use the idiom \"{idiom}\"? Provide detailed feedback in simple language" 
    )
) 
evaluation_chain = LLMChain(llm=llm, prompt = evaluation_prompt)

#  Ask user to pick an idiom and write a sentence
idiom_chosen = input("\nChoose one idiom from above to use: ")
user_example = input(f"Write a sentence using the idiom '{idiom_chosen}': ")

# Evaluate the sentence
evaluation_response = evaluation_chain.run({
    "user_example": user_example,
    "idiom": idiom_chosen
})
print("\nEvaluation:\n", evaluation_response)



Evaluation:
 The sentence "Einen Elefanten im Porzellanladen haben" is almost correct, but it is missing a verb to form a complete sentence. The idiom "einen Elefanten im Porzellanladen haben" literally means "to have an elephant in a china shop" in English, which is a metaphor for causing a lot of damage or chaos without intending to.

To use this idiom correctly in a sentence, you could say "Er hat einen Elefanten im Porzellanladen" (He has an elephant in the china shop), which means that he is causing a lot of problems or chaos without meaning to.

So, the user should add a verb to the sentence to make it complete and grammatically correct. For example, "Er hat einen Elefanten im Porzellanladen - er ist viel zu grob mit den Sachen umgegangen" (He has an elephant in the china shop - he is being much too rough with the things).


### Training

In [ ]:
# Step 2: prepare a NEW prompt template
matching_prompt = PromptTemplate(
    input_variables=["idioms"],
    template=(
        "Create a matching exercise for these German idioms and their context meaning in English.\n"
        "Idioms:\n{idioms}\n\n"
        "Return two numbered lists: one with German idioms, one with English meanings in random order."
    )
)

# Step 3: create a NEW chain for the matching exercise
matching_chain = LLMChain(llm=llm, prompt=matching_prompt)

# Step 4: format idioms for prompt
german_idioms = [idiom.split(" - ")[0].strip() for idiom in idioms]
idioms_text = "\n".join(f"{i+1}. {idiom}" for i, idiom in enumerate(german_idioms))
prompt_input = {"idioms": idioms_text}

# Step 5: run the new chain
exercise_output = matching_chain.run(prompt_input)

print(exercise_output)

German Idioms:
1. Das ist ein elefantengrünes Geschenkpapier
2. Wie die Lemminge ins Meer stürzen
3. Ein schwarzes Schaf in der Familie sein

English Meanings:
A. To act or behave in the same way as someone else, especially in a foolish or dangerous way
B. To be the black sheep of the family
C. To wrap something up in elephant green wrapping paper (said when someone is trying to hide or disguise something obvious or conceal their true intentions)
